# Sliding Window & Two Pointers — Study Guide

**Learning order:**

```
Part 0 — Foundation
  └── Core mechanisms → When to use → Two Pointers vs Sliding Window comparison
Part 1 — Two Pointers
  └── Opposite ends → Same direction → Fast & Slow pointers
Part 2 — Sliding Window
  └── Fixed window → Variable window → Window with auxiliary structure
Part 3 — Decision Guide
```

Each section introduces the concept first, then shows the pseudo-code immediately after.

---
# Part 0 — Foundation

## Why these two topics are in the same guide

Two Pointers is the foundation. Sliding Window is a specific application of Two Pointers where the two indices define a **window** and you maintain some property about the elements inside it.

Understanding Two Pointers first makes Sliding Window feel like a natural extension rather than a separate technique.

---
## Core mechanisms

**Two Pointers** uses two indices to scan a sequence. The indices move independently based on some condition — toward each other, in the same direction, or at different speeds.

**Sliding Window** uses two indices (`left` and `right`) that define a contiguous subarray. `right` expands the window, `left` shrinks it. The goal is to maintain a valid window while optimizing for size, sum, or count.

Both techniques exist to avoid nested loops — turning O(n²) brute force into O(n).

---
## When to use each

| Signal words | Likely technique |
| :--- | :--- |
| "sorted array", "pair that sums to target", "remove duplicates" | Two Pointers (opposite ends) |
| "cycle in linked list", "find middle", "detect duplicate" | Two Pointers (fast & slow) |
| "move zeros", "partition array", "Dutch national flag" | Two Pointers (same direction) |
| "subarray of length k", "fixed size window" | Sliding Window (fixed) |
| "longest/shortest subarray with condition", "minimum window" | Sliding Window (variable) |
| "at most k distinct", "no repeating characters" | Sliding Window (with hashmap) |

---
## Two Pointers vs Sliding Window — Side-by-side Comparison

| | Two Pointers | Sliding Window |
| :--- | :--- | :--- |
| **Core idea** | Two indices scanning a sequence | Two indices defining a contiguous window |
| **Movement** | Independent — toward each other, same direction, or different speeds | Coordinated — `right` always expands, `left` shrinks when invalid |
| **Input requirement** | Often sorted (opposite ends); unsorted ok for fast/slow | Unsorted ok — works on any array or string |
| **What you track** | A condition between the two pointer values | A property of elements inside the window |
| **Auxiliary structure** | Rarely needed | Often needs a hashmap or counter |
| **Typical answer** | A pair, index, or boolean | A length, count, or substring |
| **Relationship** | Foundation | Specialization of Two Pointers |

**The key distinction:** in Two Pointers the pointers move based on a comparison **between** them. In Sliding Window the pointers move based on a property **of the window contents**.

---
# Part 1 — Two Pointers

**Core concept:** place two indices into a sequence and move them based on a condition. The goal is to avoid re-examining elements you've already ruled out.

There are three movement patterns, each suited to a different class of problems.

---
## Opposite Ends

**When to use:** the array is sorted and you're looking for a pair (or triplet) that satisfies a condition. One pointer starts at the left, one at the right. You move them toward each other based on whether the current sum/product is too large or too small.

**Key insight:** because the array is sorted, moving `left` right increases the sum, moving `right` left decreases it. This lets you eliminate one element per step — O(n) instead of O(n²).

**Duplicate handling trick:** after finding a valid pair, skip all duplicate values before moving on, otherwise you'll count the same pair multiple times.

In [ ]:
# Template — Two Pointers (opposite ends)
def two_pointers_opposite(arr, target):
    left, right = 0, len(arr) - 1
    while left < right:
        current = arr[left] + arr[right]
        if current == target:
            # found a valid pair — record answer
            left += 1
            right -= 1
        elif current < target:
            left += 1       # sum too small — move left pointer right
        else:
            right -= 1      # sum too large — move right pointer left


# Example: LC 167 — Two Sum II (sorted array)
def twoSum(numbers, target):
    left, right = 0, len(numbers) - 1
    while left < right:
        s = numbers[left] + numbers[right]
        if s == target:   return [left + 1, right + 1]
        elif s < target:  left += 1
        else:             right -= 1


# Example: LC 15 — 3Sum
# Fix one element, run two pointers on the rest
def threeSum(nums):
    nums.sort()
    res = []
    for i in range(len(nums) - 2):
        if i > 0 and nums[i] == nums[i-1]: continue    # skip duplicate fixed element
        left, right = i + 1, len(nums) - 1
        while left < right:
            s = nums[i] + nums[left] + nums[right]
            if s == 0:
                res.append([nums[i], nums[left], nums[right]])
                while left < right and nums[left]  == nums[left+1]:  left += 1   # skip duplicates
                while left < right and nums[right] == nums[right-1]: right -= 1  # skip duplicates
                left += 1; right -= 1
            elif s < 0: left += 1
            else:       right -= 1
    return res

**Practice problems:**

| Problem | Key idea | Notes |
| :--- | :--- | :--- |
| LC 167 — Two Sum II | Sorted array, move based on sum vs target | Return 1-indexed |
| LC 15 — 3Sum | Fix one element, two pointers on rest | Sort first; skip duplicates at both levels |
| LC 11 — Container With Most Water | Maximize area — always move the shorter side | Moving the taller side can never increase area |

---
## Same Direction

**When to use:** you need to partition, filter, or overwrite elements in-place. Both pointers start at the left. A `slow` pointer tracks the position of the last valid element written; a `fast` pointer scans ahead looking for the next valid element.

**Key insight:** `slow` only advances when you've confirmed the current element is valid and written it into position. `fast` advances unconditionally every iteration.

In [ ]:
# Template — Two Pointers (same direction)
def two_pointers_same(arr):
    slow = 0
    for fast in range(len(arr)):
        if IS_VALID(arr[fast]):         # fast finds valid element
            arr[slow] = arr[fast]       # write to slow position
            slow += 1                   # advance slow
    return slow                         # slow = length of valid portion


# Example: LC 283 — Move Zeroes
# Move all zeros to the end, keep relative order of non-zeros
def moveZeroes(nums):
    slow = 0
    for fast in range(len(nums)):
        if nums[fast] != 0:             # valid = non-zero
            nums[slow] = nums[fast]
            slow += 1
    while slow < len(nums):             # fill remaining with zeros
        nums[slow] = 0
        slow += 1


# Example: LC 26 — Remove Duplicates from Sorted Array
def removeDuplicates(nums):
    slow = 1
    for fast in range(1, len(nums)):
        if nums[fast] != nums[fast - 1]:    # new unique element found
            nums[slow] = nums[fast]
            slow += 1
    return slow

**Practice problems:**

| Problem | Key idea | Notes |
| :--- | :--- | :--- |
| LC 283 — Move Zeroes | Slow tracks write position; fast scans | Fill remaining positions with 0 after loop |
| LC 26 — Remove Duplicates from Sorted Array | Slow = last unique written; skip duplicates | Return `slow`, not `slow - 1` |
| LC 75 — Sort Colors (Dutch Flag) | Three pointers: low, mid, high | Extend of same-direction pattern with 3 regions |

---
## Fast & Slow Pointers

**When to use:** problems on linked lists or circular sequences where you need to detect a cycle, find the middle, or find the entry point of a cycle. `fast` moves two steps per iteration, `slow` moves one.

**Key insight:** if a cycle exists, `fast` will eventually lap `slow` and they will meet inside the cycle. If no cycle exists, `fast` reaches null first.

**Finding the middle trick:** when `fast` reaches the end, `slow` is exactly at the middle — useful for problems that require splitting a linked list.

In [ ]:
# Template — Fast & Slow Pointers
def fast_slow(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next            # move one step
        fast = fast.next.next       # move two steps
        if slow == fast:            # cycle detected
            return True
    return False                    # fast reached end — no cycle


# Example: LC 141 — Linked List Cycle
def hasCycle(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
        if slow == fast: return True
    return False


# Example: LC 142 — Linked List Cycle II (find cycle entry)
# Phase 1: detect cycle. Phase 2: find entry point.
def detectCycle(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
        if slow == fast:                # cycle detected
            slow = head                 # reset slow to head
            while slow != fast:         # move both one step until they meet
                slow = slow.next
                fast = fast.next
            return slow                 # entry point of cycle
    return None


# Example: LC 876 — Middle of Linked List
def middleNode(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    return slow                         # slow is at the middle when fast reaches end

**Practice problems:**

| Problem | Key idea | Notes |
| :--- | :--- | :--- |
| LC 141 — Linked List Cycle | fast laps slow if cycle exists | Return True when slow == fast |
| LC 142 — Linked List Cycle II | Reset slow to head after detection | Distance from head to entry = distance from meeting point to entry |
| LC 876 — Middle of Linked List | slow is at middle when fast hits end | If even length, returns second middle |

---
### Two Pointers — Pattern Summary

| Pattern | Input | Both pointers start | Movement | Problems |
| :--- | :--- | :--- | :--- | :--- |
| Opposite ends | Sorted array | Left end & right end | Toward each other | 167, 15, 11 |
| Same direction | Any array | Left end | fast unconditional; slow on valid | 283, 26, 75 |
| Fast & slow | Linked list / cycle | Head | fast = 2 steps, slow = 1 step | 141, 142, 876 |

---
# Part 2 — Sliding Window

**Core concept:** maintain a contiguous window `[left, right]` over a sequence. `right` expands the window by one each iteration. `left` shrinks the window when the window becomes invalid. The answer is derived from the window size or contents at each valid state.

**The expand-then-shrink loop is the backbone of every sliding window problem:**
```
for right in range(n):
    add arr[right] to window
    while window is invalid:
        remove arr[left] from window
        left += 1
    update answer from current valid window
```

There are three variants depending on whether the window size is fixed or variable, and whether you need an auxiliary structure to track window contents.

---
## Fixed Window

**When to use:** the problem specifies an exact window size `k`. You slide a window of exactly `k` elements across the array and compute something at each position.

**Key insight:** instead of recomputing the window from scratch each step, subtract the element leaving the left and add the element entering the right — O(1) per step.

**Setup:** build the first window of size `k` before the loop, then slide from index `k` onward.

In [ ]:
# Template — Fixed Window
def fixed_window(arr, k):
    window_val = sum(arr[:k])           # build first window
    res = window_val
    for i in range(k, len(arr)):
        window_val += arr[i]            # add incoming element
        window_val -= arr[i - k]        # remove outgoing element
        res = max(res, window_val)      # update answer
    return res


# Example: LC 643 — Maximum Average Subarray I
def findMaxAverage(nums, k):
    window_sum = sum(nums[:k])
    max_sum = window_sum
    for i in range(k, len(nums)):
        window_sum += nums[i] - nums[i - k]
        max_sum = max(max_sum, window_sum)
    return max_sum / k


# Example: LC 1343 — Number of Subarrays of Size K with Average >= threshold
def numOfSubarrays(arr, k, threshold):
    window_sum = sum(arr[:k])
    count = 1 if window_sum / k >= threshold else 0
    for i in range(k, len(arr)):
        window_sum += arr[i] - arr[i - k]
        if window_sum / k >= threshold:
            count += 1
    return count

**Practice problems:**

| Problem | Key idea | Notes |
| :--- | :--- | :--- |
| LC 643 — Maximum Average Subarray I | Slide sum, divide by k at each step | Build first window before loop |
| LC 1343 — Subarrays with Average >= Threshold | Fixed window sum vs threshold | Same pattern as 643 |
| LC 567 — Permutation in String | Fixed window of size `len(s1)`, compare char counts | Use a frequency counter; fixed window with hashmap |

---
## Variable Window

**When to use:** the window size is not fixed — you expand until the window becomes invalid, then shrink from the left until it becomes valid again. The answer is the maximum or minimum valid window size seen.

**Key insight:** write the validity condition clearly before coding. The `while` condition that shrinks the window is the entire logic of the problem — get this right first.

**Two flavors:**
- **Longest valid window** — update answer after shrinking (`right - left + 1` at each step)
- **Shortest valid window** — update answer inside the shrink loop (every time window is still valid after shrinking)

In [ ]:
# Template — Variable Window (longest valid)
def variable_window_longest(arr):
    left = 0
    res = 0
    window = {}                             # track window contents
    for right in range(len(arr)):
        # expand: add arr[right] to window
        window[arr[right]] = window.get(arr[right], 0) + 1

        while WINDOW_IS_INVALID(window):    # shrink until valid
            window[arr[left]] -= 1
            if window[arr[left]] == 0: del window[arr[left]]
            left += 1

        res = max(res, right - left + 1)    # update after shrinking
    return res


# Template — Variable Window (shortest valid)
def variable_window_shortest(arr, target):
    left = 0
    res = float('inf')
    window_sum = 0
    for right in range(len(arr)):
        window_sum += arr[right]            # expand
        while window_sum >= target:         # shrink while still valid
            res = min(res, right - left + 1)  # update INSIDE shrink loop
            window_sum -= arr[left]
            left += 1
    return res if res != float('inf') else 0


# Example: LC 3 — Longest Substring Without Repeating Characters
def lengthOfLongestSubstring(s):
    left = 0
    seen = {}
    res = 0
    for right in range(len(s)):
        seen[s[right]] = seen.get(s[right], 0) + 1
        while seen[s[right]] > 1:          # invalid: duplicate char in window
            seen[s[left]] -= 1
            left += 1
        res = max(res, right - left + 1)
    return res


# Example: LC 209 — Minimum Size Subarray Sum
def minSubArrayLen(target, nums):
    left = 0
    window_sum = 0
    res = float('inf')
    for right in range(len(nums)):
        window_sum += nums[right]
        while window_sum >= target:         # valid: update answer inside shrink
            res = min(res, right - left + 1)
            window_sum -= nums[left]
            left += 1
    return res if res != float('inf') else 0

**Practice problems:**

| Problem | Flavor | Validity condition | Notes |
| :--- | :--- | :--- | :--- |
| LC 3 — Longest Substring Without Repeating Characters | Longest | No char appears more than once | Update answer after shrink loop |
| LC 209 — Minimum Size Subarray Sum | Shortest | Window sum >= target | Update answer inside shrink loop |
| LC 424 — Longest Repeating Character Replacement | Longest | `window_size - max_freq <= k` | Track max frequency of any char in window |

---
## Window with Auxiliary Structure

**When to use:** the validity condition depends on the **contents** of the window, not just its size or sum. You need a hashmap or counter to track what's inside the window efficiently.

**Key insight:** define what "valid" means in terms of your auxiliary structure before writing any code. For example: "valid = all characters of `t` are covered" or "valid = at most `k` distinct characters".

**The `have` vs `need` pattern:** track how many unique characters you currently satisfy (`have`) vs how many you need (`need`). This avoids iterating over the counter to check validity at every step — O(1) validity check.

In [ ]:
from collections import Counter

# Example: LC 76 — Minimum Window Substring
# Find shortest window in s that contains all chars of t
def minWindow(s, t):
    if not t or not s: return ""

    need    = Counter(t)                # frequency of each char needed
    have    = 0                         # how many chars currently satisfied
    need_count = len(need)              # number of unique chars to satisfy
    window  = {}                        # frequency of chars in current window

    left = 0
    res  = float('inf')
    res_left = res_right = 0

    for right in range(len(s)):
        c = s[right]
        window[c] = window.get(c, 0) + 1
        if c in need and window[c] == need[c]:  # this char is now fully satisfied
            have += 1

        while have == need_count:               # valid window — try to shrink
            if right - left + 1 < res:
                res = right - left + 1
                res_left, res_right = left, right
            window[s[left]] -= 1
            if s[left] in need and window[s[left]] < need[s[left]]:
                have -= 1                       # lost a satisfied char
            left += 1

    return s[res_left:res_right+1] if res != float('inf') else ""


# Example: LC 340 — Longest Substring with At Most K Distinct Characters
def lengthOfLongestSubstringKDistinct(s, k):
    window = {}
    left = 0
    res  = 0
    for right in range(len(s)):
        window[s[right]] = window.get(s[right], 0) + 1
        while len(window) > k:                  # invalid: too many distinct chars
            window[s[left]] -= 1
            if window[s[left]] == 0:
                del window[s[left]]
            left += 1
        res = max(res, right - left + 1)
    return res

**Practice problems:**

| Problem | Auxiliary structure | Validity condition | Notes |
| :--- | :--- | :--- | :--- |
| LC 76 — Minimum Window Substring | Counter + `have`/`need` | `have == need_count` | Hardest sliding window — master this last |
| LC 340 — Longest Substring K Distinct | Hashmap of char counts | `len(window) <= k` | Delete key when count hits 0 |
| LC 438 — Find All Anagrams | Fixed window + Counter comparison | Window counter matches `p` counter | Fixed window variant with hashmap |

---
### Sliding Window — Pattern Summary

| Pattern | Window size | Answer location | Auxiliary structure | Problems |
| :--- | :--- | :--- | :--- | :--- |
| Fixed | Exactly k | After each slide | Optional counter | 643, 567, 1343 |
| Variable (longest) | Flexible | After shrink loop | Optional hashmap | 3, 424 |
| Variable (shortest) | Flexible | Inside shrink loop | Optional hashmap | 209, 76 |
| With auxiliary structure | Flexible | Inside shrink loop | Counter + have/need | 76, 340, 438 |

---
# Part 3 — Decision Guide

```
Is the input a linked list?
├── Yes → Fast & Slow Pointers
│         ├── Detect cycle?               → LC 141
│         ├── Find cycle entry?           → LC 142 (reset slow to head after meeting)
│         └── Find middle?               → LC 876 (slow is at middle when fast hits end)
│
└── No  → Array or string
          │
          ├── Is the array sorted AND looking for a pair/triplet?
          │   └── Yes → Two Pointers (opposite ends)
          │             ├── Two elements summing to target   → LC 167
          │             ├── Three elements summing to target → LC 15 (fix one, run two pointers)
          │             └── Maximize area / distance        → LC 11 (move shorter side)
          │
          ├── Need to filter or partition in-place?
          │   └── Yes → Two Pointers (same direction)
          │             ├── slow = write position; fast = read position
          │             └── LC 283, 26, 75
          │
          └── Looking for a subarray or substring?
              └── Yes → Sliding Window
                        │
                        ├── Window size is fixed (k given explicitly)
                        │   └── Fixed Window — build first window, then slide
                        │       LC 643, 567, 1343
                        │
                        ├── Find LONGEST subarray satisfying a condition
                        │   └── Variable Window (longest) — update answer AFTER shrink
                        │       LC 3, 424
                        │
                        └── Find SHORTEST subarray satisfying a condition
                            └── Variable Window (shortest) — update answer INSIDE shrink
                                └── Need to track window contents? → add Counter/hashmap
                                    LC 209, 76, 340
```